In [1]:
#from google.colab import drive
import os
import pandas as pd

# Inizializzazione Google Drive
#try:
#    drive.mount('/content/drive')
#except:
#    pass

path = "/kaggle/input/datasets/jeanmar1728/report-originali-key/report_originali.csv"
df = pd.read_csv(path)

In [2]:
df

,Unnamed: 0,paese,periodo,inizio_periodo,fine_periodo,nome_file,testo_originale
0,0,Democratic Republic of the Congo,Dec 2012 / Dec 2012,Dec 2012,Dec 2012,Democratic_Republic_of_the_Congo_Dec_2012_-_De...,The 8th analysis cycle on the Integrated Food ...
1,1,Burundi,Nov 2024 / Mar 2025,Nov 2024,Mar 2025,Burundi_Nov_2024_-_Mar_2025_KeyResults.txt,"Between January and March 2025, which coincide..."
2,2,Madagascar,Aug 2017 / Mar 2018,Aug 2017,Mar 2018,Madagascar_Aug_2017_-_Mar_2018_KeyResults.txt,The period from August to October 2017 coincid...
3,3,Yemen,Oct 2023 / Feb 2024,Oct 2023,Feb 2024,Yemen_Oct_2023_-_Feb_2024_KeyResults.txt,The updated analysis indicates that for the pe...
4,4,Mozambique,Apr 2018 / Sep 2018,Apr 2018,Sep 2018,Mozambique_Apr_2018_-_Sep_2018_KeyResults.txt,DISCLAIMER: please note that this IPC Acute Fo...
...,...,...,...,...,...,...,...
492,492,Lesotho,Jul 2022 / Mar 2023,Jul 2022,Mar 2023,Lesotho_Jul_2022_-_Mar_2023_KeyResults.txt,According to the latest results of an IPC Acut...
493,493,Zimbabwe,May 2016 / Mar 2017,May 2016,Mar 2017,Zimbabwe_May_2016_-_Mar_2017_KeyResults.txt,"In the period from April-June 2016, the countr..."
494,494,Djibouti,May 2013 / May 2013,May 2013,May 2013,Djibouti_May_2013_-_May_2013_KeyResults.txt,Food availability in the Republic of Djibouti ...
495,495,Uganda,Jul 2024 / Jun 2025,Jul 2024,Jun 2025,Uganda_Jul_2024_-_Jun_2025_KeyResults.txt,"Approximately 797,000 people in refugee-hostin..."


In [3]:
!pip install -q gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 5.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 85.0 MB/s eta 0:00:00:00:0100:01


In [6]:
import pandas as pd
from gliner import GLiNER
import spacy
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Caricamento modelli
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")
gliner_model = gliner_model.to(device)
gliner_model.eval() 

nlp_model = spacy.load("en_core_web_sm", disable=["ner", "parser"]) # Velocizza spaCy
nlp_model.add_pipe("sentencizer")

# Mappatura dinamica per delegare a GLiNER SOLO le entità spaziali
tag_mapping = {
    "country": "[AFFECTED_AREA]",
    "region": "[AFFECTED_AREA]",
    "province": "[AFFECTED_AREA]",
    "district": "[AFFECTED_AREA]",
    "city": "[AFFECTED_AREA]",
    "village": "[AFFECTED_AREA]",
    "location": "[AFFECTED_AREA]",
    "landmark": "[AFFECTED_AREA]",
    "year": "[DATE]",
    "month": "[DATE]",
    "date": "[DATE]"
}
labels = list(tag_mapping.keys())


# =====================================================================
# 2. FUNZIONI CORE DI PREPROCESSING E ABLAZIONE
# =====================================================================
def split_into_sentences(text: str, nlp: spacy.Language) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


def anonymize_sentence(sentence: str, model: GLiNER) -> str:
    # Ablazione Spaziale con GLiNER
    entities = model.predict_entities(sentence, labels, threshold=0.35, max_len=512)
    entities_sorted = sorted(entities, key=lambda x: x['start'], reverse=True)

    anonymized = sentence
    for ent in entities_sorted:
        start = ent['start']
        end = ent['end']
        placeholder = tag_mapping.get(ent['label'], "[REDACTED]")
        anonymized = anonymized[:start] + placeholder + anonymized[end:]

    
    return anonymized


# =====================================================================
# 3. ORCHESTRAZIONE E INTEGRAZIONE PANDAS
# =====================================================================
def anonymize_full_report(full_text: str, model: GLiNER, nlp: spacy.Language) -> str:
    if not isinstance(full_text, str) or not full_text.strip():
        return full_text

    # Il modello nlp deve essere iniettato esplicitamente
    sentences = split_into_sentences(full_text, nlp)

    anonymized_sentences = [anonymize_sentence(s, model) for s in sentences]

    return " ".join(anonymized_sentences)


def process_dataframe_safe(df: pd.DataFrame, input_col: str, output_col: str) -> pd.DataFrame:
    print(f"Elaborazione di {len(df)} report con gestione automatica della lunghezza frasi...")

    df[output_col] = df[input_col].apply(
        lambda x: anonymize_full_report(x, gliner_model, nlp_model)
    )

    return df

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [7]:
df_processed = process_dataframe_safe(df, input_col="testo_originale", output_col="report_anonimo")
df_processed

Elaborazione di 497 report con gestione automatica della lunghezza frasi...


,Unnamed: 0,paese,periodo,inizio_periodo,fine_periodo,nome_file,testo_originale,report_anonimo
0,0,Democratic Republic of the Congo,Dec 2012 / Dec 2012,Dec 2012,Dec 2012,Democratic_Republic_of_the_Congo_Dec_2012_-_De...,The 8th analysis cycle on the Integrated Food ...,The 8th analysis cycle on the Integrated Food ...
1,1,Burundi,Nov 2024 / Mar 2025,Nov 2024,Mar 2025,Burundi_Nov_2024_-_Mar_2025_KeyResults.txt,"Between January and March 2025, which coincide...","Between [DATE] and [DATE] [DATE], which coinci..."
2,2,Madagascar,Aug 2017 / Mar 2018,Aug 2017,Mar 2018,Madagascar_Aug_2017_-_Mar_2018_KeyResults.txt,The period from August to October 2017 coincid...,The period from [DATE] to [DATE] [DATE] coinci...
3,3,Yemen,Oct 2023 / Feb 2024,Oct 2023,Feb 2024,Yemen_Oct_2023_-_Feb_2024_KeyResults.txt,The updated analysis indicates that for the pe...,The updated analysis indicates that for the pe...
4,4,Mozambique,Apr 2018 / Sep 2018,Apr 2018,Sep 2018,Mozambique_Apr_2018_-_Sep_2018_KeyResults.txt,DISCLAIMER: please note that this IPC Acute Fo...,DISCLAIMER: please note that this IPC Acute Fo...
...,...,...,...,...,...,...,...,...
492,492,Lesotho,Jul 2022 / Mar 2023,Jul 2022,Mar 2023,Lesotho_Jul_2022_-_Mar_2023_KeyResults.txt,According to the latest results of an IPC Acut...,According to the latest results of an IPC Acut...
493,493,Zimbabwe,May 2016 / Mar 2017,May 2016,Mar 2017,Zimbabwe_May_2016_-_Mar_2017_KeyResults.txt,"In the period from April-June 2016, the countr...","In the period from [DATE] [DATE], the [AFFECTE..."
494,494,Djibouti,May 2013 / May 2013,May 2013,May 2013,Djibouti_May_2013_-_May_2013_KeyResults.txt,Food availability in the Republic of Djibouti ...,Food availability in the [AFFECTED_AREA] is ma...
495,495,Uganda,Jul 2024 / Jun 2025,Jul 2024,Jun 2025,Uganda_Jul_2024_-_Jun_2025_KeyResults.txt,"Approximately 797,000 people in refugee-hostin...","Approximately 797,000 people in [AFFECTED_AREA..."


In [8]:
df_processed['testo_originale'][0]

'The 8th analysis cycle on the Integrated Food Security Classification Framework (IPC) of DRC held in December 2012 identified 6.4 million people affected by a situation of food and livelihood crises, 77 regions have been classified in phase 3 and 8 regions in Phase 4 throughout DRC. The affected areas are in need of an emergency food and agricultural assistance. Regions in humanitarian emergency situation  (phase 4) are located in areas affected by armed conflicts of Northern Kivu (Rutshuru, Masisi), of Southern Kivu (Kalehe, Shabunda), of Maniema (Pangi) and Katanga (Mitwaba, Manono, Pweto), whereas all the provinces in DRC  are  taken  into  account  in  phase  3.  Established on  the  basis  of  a multidimensional  analysis  of  food  security,  this  classification  is  based  on  food security indicators which are household food consumption, livelihoods evolution, and nutritional state of children from 6-59 months and the mortality rate.\nAs  compared  to  the  7th IPC  analysis 

In [9]:
df_processed['report_anonimo'][0]

'The 8th analysis cycle on the Integrated Food Security Classification Framework (IPC) of [AFFECTED_AREA] held in [DATE] identified 6.4 million people affected by a situation of food and livelihood crises, [AFFECTED_AREA] have been classified in phase 3 and [AFFECTED_AREA] in Phase 4 throughout [AFFECTED_AREA]. The [AFFECTED_AREA] are in need of an emergency food and agricultural assistance. [AFFECTED_AREA]  (phase 4) are located in areas affected by armed conflicts of [AFFECTED_AREA] ([AFFECTED_AREA], [AFFECTED_AREA]), of [AFFECTED_AREA] ([AFFECTED_AREA], [AFFECTED_AREA]), of [AFFECTED_AREA] ([AFFECTED_AREA]) and [AFFECTED_AREA] ([AFFECTED_AREA], [AFFECTED_AREA], [AFFECTED_AREA]), whereas all the provinces in [AFFECTED_AREA]  are  taken  into  account  in  phase  3. Established on  the  basis  of  a multidimensional  analysis  of  food  security,  this  classification  is  based  on  food security indicators which are household food consumption, livelihoods evolution, and nutritional 

In [10]:
df_processed.to_csv("/kaggle/working/report_anonimizzati.csv", index = False)